# Stage 4 — Feature engineering & modeling

From *who churned* to *who resembles the customers who churned*.

Two rules govern this notebook:

1. **The test set is not touched.** Every comparison below is 5-fold cross-validation inside the
   training set. The test set is scored once, in Stage 5.
2. **Nothing is fitted before the split** — no scaler, no encoder, no bin edges.

**What this dataset supports, stated once.** There are no event dates and no observation window;
`Churn` means *left within the last month*. So there is **no defined prediction point**. A model
here separates customers who have already churned from those who have not, using current
attributes. That is legitimate risk scoring and it is what Stage 6 needs — but it is not
forecasting, and the output is not "probability of churning next month".

## Setup

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate, cross_val_predict
from sklearn.pipeline import Pipeline

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from features import load_modelling_data, split, make_preprocessor, NUMERIC_FEATURES
from modeling import ContractRule, score_model, CV, SCORING

pd.set_option("display.width", 200)

data = load_modelling_data()
X_train, X_test, y_train, y_test = split(data)

7,032 customers for modelling (11 with tenure = 0 excluded)
train 5,274 rows, churn 0.266
test  1,758 rows, churn 0.266   (untouched until Stage 5)


The 11 customers with `tenure = 0` are removed **before** the split. Their `Churn = "No"` is true by
definition — no completed billing period — so they carry no signal, and leaving them in would place
invalid labels in whichever fold they fell into.

## Baselines

Two, and the comparison between them is the point.

In [2]:
baselines = [
    score_model(DummyClassifier(strategy="most_frequent"), X_train, y_train, "Predict nobody churns", SCORING),
    score_model(ContractRule(), X_train, y_train, "Rule: month-to-month", SCORING),
]
pd.DataFrame(baselines).set_index("model")[["accuracy", "precision", "recall", "f1"]].round(3)

,accuracy,precision,recall,f1
model,,,,
Predict nobody churns,0.734,0.000,0.000,0.000
Rule: month-to-month,0.655,0.428,0.885,0.577


**The more accurate baseline is the useless one.** "Predict nobody churns" scores 73.4% accuracy and
flags zero customers — there is nobody for a retention team to contact. The rule gives up 8 points of
accuracy and catches **88.5%** of churners.

The bar for the models is therefore **F1 = 0.577**, not 73.4% accuracy.

Note also `precision = 0.428`: the rule flags 3,875 month-to-month customers, and 57% of them would
not have churned. Reducing that waste is what a model has to earn its place with.

## Feature variants

Stages 2–3 left four decisions open. Cross-validation settles them.

In [3]:
def logreg(numeric, bin_tenure=False, **kwargs):
    return Pipeline([("prep", make_preprocessor("linear", numeric, bin_tenure)),
                     ("model", LogisticRegression(max_iter=2000, **kwargs))])

variants = [
    ("A. all 3 numerics (naive)",  logreg(NUMERIC_FEATURES)),
    ("B. drop MonthlyCharges",     logreg(["tenure", "TotalCharges"])),
    ("C. drop Monthly + Total",    logreg(["tenure"])),
    ("D. C + tenure binned",       logreg(["tenure"], bin_tenure=True)),
]
results = pd.DataFrame([score_model(m, X_train, y_train, n) for n, m in variants]).set_index("model")
results[["accuracy", "precision", "recall", "f1", "roc_auc", "pr_auc", "roc_auc_std"]].round(3)

,accuracy,precision,recall,f1,roc_auc,pr_auc,roc_auc_std
model,,,,,,,
A. all 3 numerics (naive),0.801,0.650,0.544,0.592,0.846,0.662,0.020
B. drop MonthlyCharges,0.800,0.649,0.543,0.591,0.846,0.662,0.020
C. drop Monthly + Total,0.800,0.648,0.538,0.588,0.844,0.660,0.020
D. C + tenure binned,0.806,0.666,0.540,0.596,0.848,0.666,0.022


**The four variants are indistinguishable.** ROC-AUC spans 0.844–0.848, while the fold-to-fold
standard deviation is **0.020** — five times the spread between variants. None of these differences
are real.

That is the useful result: the feature decisions have **no measurable effect on prediction**, so
they can be made on interpretability at no cost.

**Chosen: variant C.** Simplest (21 encoded features), lowest collinearity, same performance.

- Dropping `MonthlyCharges` — VIF **878**, because it is a deterministic sum of the service columns
  — moved ROC-AUC from 0.846 to 0.844. Free.
- `TotalCharges` adds nothing and carries VIF 9.5.
- Binning `tenure` (variant D) is nominally best by 0.004 but adds five features and has the highest
  fold variance. Complexity buying noise.

Binning was done with `KBinsDiscretizer` **inside** the pipeline, so each fold learns its own edges.
Using `pd.qcut` on the full data beforehand — the obvious way — would set the cut points using
held-out rows. A quiet leak, and an easy one to miss.

## Class imbalance

26.6% positive. The standard moves are resampling or class weights; both are rejected here, and the
reason is what happens downstream.

In [4]:
weighted = [score_model(logreg(["tenure"]), X_train, y_train, "C, natural distribution"),
            score_model(logreg(["tenure"], class_weight="balanced"), X_train, y_train, "C, class_weight=balanced")]
print(pd.DataFrame(weighted).set_index("model")[["precision", "recall", "f1", "roc_auc", "pr_auc"]].round(3).to_string())

# Is it a better model, or the same model at a different threshold?
p_natural = cross_val_predict(logreg(["tenure"]), X_train, y_train, cv=CV, method="predict_proba")[:, 1]
p_balanced = cross_val_predict(logreg(["tenure"], class_weight="balanced"), X_train, y_train,
                               cv=CV, method="predict_proba")[:, 1]

print(f"\nSpearman correlation of the two rankings : "
      f"{pd.Series(p_natural).corr(pd.Series(p_balanced), method='spearman'):.4f}")
print(f"mean predicted probability   natural {p_natural.mean():.3f} | balanced {p_balanced.mean():.3f}")
print(f"actual churn rate                     {y_train.mean():.3f}")

                          precision  recall     f1  roc_auc  pr_auc
model                                                              
C, natural distribution       0.648   0.538  0.588    0.844   0.660
C, class_weight=balanced      0.522   0.795  0.630    0.844   0.659



Spearman correlation of the two rankings : 0.9995
mean predicted probability   natural 0.266 | balanced 0.414
actual churn rate                     0.266


F1 improves from 0.588 to 0.630 — but **ROC-AUC and PR-AUC are unchanged and the two rankings
correlate at 0.9995.** It is not a better model. It is the same model with the threshold moved.

The cost is real: natural training gives a mean predicted probability of **0.266, exactly the base
rate**. Balanced inflates it to **0.414**.

Stage 6 ranks customers by `churn probability × customer value`. A probability that overstates risk
by 15 points would corrupt every ranking. We keep calibration and move the threshold in Stage 5,
which gets the same benefit for free.

SMOTE is rejected for the same reason, more strongly — it invents minority examples and distorts the
base rate the model is supposed to learn.

## Gradient boosting

Does a flexible model find non-linearities and interactions the linear one misses?

In [5]:
def boosted(numeric, **kwargs):
    # No scaling: tree splits are scale-invariant, so standardising changes nothing.
    return Pipeline([("prep", make_preprocessor("tree", numeric)),
                     ("model", HistGradientBoostingClassifier(random_state=42, **kwargs))])

# HistGradientBoosting only enables early stopping above 10,000 rows. With 5,274 the
# default runs all 100 iterations unchecked, so the first row below is overfitting.
trees = [
    ("GB default",                boosted(["tenure"])),
    ("GB + early stopping",       boosted(["tenure"], early_stopping=True, validation_fraction=0.15)),
    ("GB shallow (4 leaves)",     boosted(["tenure"], max_leaf_nodes=4, learning_rate=0.05, max_iter=300)),
    ("GB shallow + all numerics", boosted(NUMERIC_FEATURES, max_leaf_nodes=4, learning_rate=0.05, max_iter=300)),
]
tree_results = pd.DataFrame([score_model(m, X_train, y_train, n) for n, m in trees]).set_index("model")
print(tree_results[["precision", "recall", "f1", "roc_auc", "pr_auc"]].round(3).to_string())
print("\nLR variant C for reference:   roc_auc 0.844   pr_auc 0.660   f1 0.588")

                           precision  recall     f1  roc_auc  pr_auc
model                                                               
GB default                     0.638   0.526  0.576    0.839   0.651
GB + early stopping            0.651   0.511  0.572    0.844   0.657
GB shallow (4 leaves)          0.659   0.529  0.587    0.850   0.670
GB shallow + all numerics      0.662   0.537  0.592    0.848   0.667

LR variant C for reference:   roc_auc 0.844   pr_auc 0.660   f1 0.588


Two things worth noting before the comparison:

- **The default model was overfitting** — constraining it to 4 leaves recovered 0.011 ROC-AUC.
  sklearn's defaults assume more data than we have.
- **The redundant numerics do not help the tree either** (0.848 vs 0.850 — inside noise). Trees
  handle redundant features gracefully, so this is a null result rather than a penalty; it simply
  confirms that `MonthlyCharges` and `TotalCharges` carry nothing the service columns lack, for
  either model. (With the *unconstrained* default they did measurably hurt, 0.839 to 0.833 — but
  that model was overfitting, so the comparison is not worth much.)

The constrained tree reaches 0.850 against the linear model's 0.844. Since the fold standard
deviation is 0.020, that gap needs a paired test rather than a glance.

In [6]:
lr_best = logreg(["tenure"])
gb_best = boosted(["tenure"], max_leaf_nodes=4, learning_rate=0.05, max_iter=300)

# Same folds for both models, so differences are paired rather than two noisy averages.
scored = {name: cross_validate(m, X_train, y_train, cv=CV, scoring=["roc_auc", "average_precision"])
          for name, m in [("LR", lr_best), ("GB", gb_best)]}

for metric, key in [("ROC-AUC", "test_roc_auc"), ("PR-AUC", "test_average_precision")]:
    difference = scored["GB"][key] - scored["LR"][key]
    print(f"{metric}: GB - LR per fold {np.round(difference, 4)}")
    print(f"   mean {difference.mean():+.4f}, GB wins {int((difference > 0).sum())}/5 folds\n")

ROC-AUC: GB - LR per fold [0.0005 0.0007 0.0042 0.0152 0.0076]
   mean +0.0056, GB wins 5/5 folds

PR-AUC: GB - LR per fold [-0.0018 -0.0075  0.0158  0.0239  0.0214]
   mean +0.0104, GB wins 3/5 folds



**ROC-AUC: consistent but tiny** — the tree wins every fold, by 0.006 on average.
**PR-AUC: inconsistent** — it loses two folds outright.

PR-AUC is the more relevant of the two here: it concentrates on the positive class, which is what a
retention list is made of. On that metric the advantage is not dependable.

### Decision: logistic regression

Not because the tree failed. It is marginally and consistently better at ranking, and pretending
otherwise would be dishonest. The reason is that **+0.006 ROC-AUC does not buy back
interpretability**, and Stage 6 has to explain why a customer was flagged.

There is also a real finding in the tie. **A flexible model finds almost nothing extra**, which is
consistent with the checks made before fitting: log-odds is close to linear in `tenure` (R² = 0.93),
and once `InternetService` is in the model `MonthlyCharges` is monotone within every tier. The
structure in this data really is close to linear in log-odds, so there is little for tree
flexibility to exploit. That is a result, not an absence of one.

## Is the model calibrated?

Stage 6 multiplies churn probability by customer value, so the probabilities must mean what they
say: customers scored at 0.6 should churn about 60% of the time.

**A warning about the obvious check.** For logistic regression with an intercept, the mean predicted
probability *always* equals the base rate on the training data. It is forced by the score equation,
holds for any such fit, and is evidence of nothing. Calibration must be checked on unseen data.

In [7]:
from sklearn.metrics import brier_score_loss

# Out-of-fold: every row scored by a model that never saw it.
p_oof = cross_val_predict(logreg(["tenure"]), X_train, y_train, cv=CV, method="predict_proba")[:, 1]

deciles = pd.DataFrame({"predicted": p_oof, "actual": y_train.to_numpy(), "bin": pd.qcut(p_oof, 10)})
calibration = deciles.groupby("bin", observed=True).agg(
    customers=("actual", "size"), predicted=("predicted", "mean"), observed=("actual", "mean"))
calibration["gap_pp"] = ((calibration["observed"] - calibration["predicted"]) * 100).round(1)
print(calibration.round(3).to_string())

base_rate = np.full(len(y_train), y_train.mean())
print(f"\nBrier {brier_score_loss(y_train, p_oof):.4f}"
      f"  (always predicting the base rate scores {brier_score_loss(y_train, base_rate):.4f})")
print(f"worst decile gap {calibration['gap_pp'].abs().max():.1f}pp")

                   customers  predicted  observed  gap_pp
bin                                                      
(0.00215, 0.0151]        528      0.010     0.013     0.4
(0.0151, 0.0338]         527      0.024     0.025     0.1
(0.0338, 0.0644]         527      0.047     0.057     1.0
(0.0644, 0.115]          528      0.087     0.085    -0.2
(0.115, 0.18]            527      0.148     0.152     0.4
(0.18, 0.267]            527      0.221     0.226     0.4
(0.267, 0.387]           528      0.324     0.347     2.3
(0.387, 0.536]           527      0.456     0.412    -4.4
(0.536, 0.669]           527      0.601     0.583    -1.9
(0.669, 0.889]           528      0.741     0.759     1.8

Brier 0.1359  (always predicting the base rate scores 0.1952)
worst decile gap 4.4pp


Predicted and observed track closely across all ten deciles — worst gap **4.4pp**, most under 2pp —
and the Brier score beats always predicting the base rate. The probabilities can be used as
probabilities, which is what Stage 6 needs.

## Fit and save

In [8]:
from train_model import build_model, MODEL_PATH

model = build_model().fit(X_train, y_train)
print(f"{model['prep'].transform(X_train).shape[1]} encoded features")
print(f"saved by src/train_model.py to {MODEL_PATH.name}")

21 encoded features
saved by src/train_model.py to churn_logistic_regression.joblib


---
# Stage 4 summary

| Decision | Outcome | Settled by |
|---|---|---|
| Population | 7,032 customers; 11 `tenure = 0` excluded | Stage 2 — label is definitional |
| Split | 75/25 stratified, before any fitting | Leakage |
| `MonthlyCharges` | **Dropped** | VIF 878; costs 0.002 ROC-AUC |
| `TotalCharges` | **Dropped** | No benefit to either model |
| `tenure` | **Linear, unbinned** | Binning adds 5 features for +0.004, higher variance |
| Imbalance | **Natural distribution** | Weighting only moves the threshold, and decalibrates |
| Model | **Logistic regression**, 21 features | Tree gains +0.006 ROC-AUC, inconsistent on PR-AUC |

**What this stage did not do:** no test-set scoring, no threshold selection, no coefficient
interpretation. Those are Stage 5, and keeping them separate is what stops the test set being tuned
against.

### Carried into Stage 5

- **The default 0.5 threshold is wrong for this problem.** The model's recall at 0.5 is 0.538 —
  *worse* than the dumb rule's 0.885. The model ranks far better (ROC-AUC 0.844 vs a single
  threshold); the cut-off simply has not been chosen yet. That choice is the main work of Stage 5.
- Probabilities are calibrated **out-of-fold** — Brier 0.136 against 0.195 for always predicting
  the base rate, worst decile gap 4.4pp. Stage 6 depends on this.
- `Contract` is the largest coefficient and may mask weaker signals — worth checking during
  interpretation.